In [ ]:
import numpy as np
import pandas as pd
from scipy.interpolate import interpolate

from DarkCapPy import *
import DarkCapPy.DarkPhoton as DP

import matplotlib.pylab as plt
import seaborn as sns

from scipy.interpolate import griddata
from skimage import measure

import os

In [ ]:
# Read in the file

# This file is for testing, assuming you ran the previous notebooks
# It only has a few mass points so it's not good for all these plots
#infilename = 'rates_muons_electrons_both_alphas_KAPPAS_FOR_TESTING.parquet'

# Bellis made this file
#infilename = 'rates_muons_electrons_both_alphas_mx_10_1000_mxstep_10.parquet'
#infilename = 'rates_muons_electrons_both_alphas_mx_10TeV_1000TeV_mxstep_10TeV_WITH_CALC_ACCEPTANCES.parquet'
#infilename = 'rates_muons_electrons_both_alphas_KAPPAS_FOR_TESTING_fine_grain_epsilon_and_mas.parquet'
#infilename = 'rates_muons_electrons_both_alphas_KAPPAS_10_100_1000_10000_100000_fine_grain_epsilon_and_mas.parquet'
#infilename = 'rates_muons_electrons_both_alphas_mx_225_500_mxstep_25_WITH_CALC_ACCEPTANCES.parquet'
infilename = 'rates_muons_electrons_both_alphas_KAPPAS_10_1000000_varying_steps_coarse_grain_epsilon_and_mas_WITH_CALC_ACCEPTANCES.parquet'

df = pd.read_parquet(infilename)

print('Columns')
print(df.columns)
print()
print('Masses: ')
print(df['mx'].unique())
print()

df

In [ ]:
#x = df['mx']
# This was an early estimate, but it is not accurate. 
y = df['angular_acceptance'].unique()
y

#plt.plot(x,y,'.')

# Plot highest rates for each mass point

In [ ]:
masses = df['mx'].unique()

for mass in masses:
    print(int(mass))


In [ ]:
max_rates = []
max_eps = []
max_ma = []

for mass in masses:
    #print(mass)
    filter = df['mx'] == mass
    dftmp = df[filter]
    dfmax = dftmp.loc[dftmp['rate_1yr'].idxmax()]
    #print(dfmax)
    max_rates.append(dfmax['rate_1yr'])
    max_eps.append(dfmax['epsilon'])
    max_ma.append(dfmax['ma'])
    
max_rates = np.array(max_rates)
max_eps = np.array(max_eps)
max_ma = np.array(max_ma)

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.plot(masses,max_rates,'o')
plt.xscale('log')
plt.yscale('log')
plt.xlabel(r'm$_{X}$ (GeV/c$^2$)', fontsize=14)
plt.ylabel(r'# at detector / year', fontsize=14)

plt.subplot(1,3,2)
plt.plot(masses,max_eps,'o')
plt.xscale('log')
plt.yscale('log')
plt.xlabel(r'm$_{X}$ (GeV/c$^2$)', fontsize=14)
plt.ylabel(r'$\epsilon$ for max rate', fontsize=14)

plt.subplot(1,3,3)
plt.plot(masses,max_ma,'o')
plt.xscale('log')
#plt.yscale('log')
plt.xlabel(r'm$_{X}$ (GeV/c$^2$)', fontsize=14)
plt.ylabel(r'm$_A$ for max rate (GeV/c$^2$)', fontsize=14)

plt.tight_layout()
plt.savefig('max_values_rates_epsilon_ma.png')

# Convert a rate from IceCube to CMS

In [ ]:
# Pick an arbitrary row

idx = 1000113

dftmp = df.iloc[idx]

dftmp



In [ ]:
rate_icecube = dftmp['rate_1yr']

volume_icecube = 1000**3 # in meters

volume_floating = dftmp['volume_m3_floating']
volume_core = dftmp['volume_m3_core']

frac_floating = dftmp['frac_ecut100_floating']
frac_core = dftmp['frac_ecut100_core']


print(f'Rate (1 year) in Icecube:                      {rate_icecube:.2f}')
print(f'Volume of Icecube (m^3):                       {volume_icecube:.2e}')

print(f'Volume of rock for floating (m^3):             {volume_floating:.2e}')
print(f'Volume of rock for core (m^3):                 {volume_core:.2e}')

print(f'Geometric acceptance (E>100 GeV)for floating:  {frac_floating:.2e}')
print(f'Geometric acceptance (E>100 GeV)for core:      {frac_core:.2e}')

rate_cms_floating = rate_icecube*(volume_floating/volume_icecube)*frac_floating
rate_cms_core = rate_icecube*(volume_core/volume_icecube)*frac_core

print(f'Rate at CMS (1 yr) for floating model:         {rate_cms_floating:.2f}')
print(f'Rate at CMS (1 yr) for core model:             {rate_cms_core:.2f}')

In [ ]:
def convert_from_icecube_to_cms_rates(df, mx, rate_icecube, ecut=100, hit_id=False, verbose=False):

    geometric_acceptance_scale = []
    volume_scale = []
    
    for dm_model in ['floating', 'core']:
        scaling_column_name = f'frac_ecut{ecut:d}_{dm_model}'
        if hit_id:
            scaling_column_name = f'frac_hit_id_ecut{ecut:d}_{dm_model}'
            
        
        scale = df[df['mx']==mx][scaling_column_name].values
                        
        if len(scale)>0:
            geometric_acceptance_scale.append(scale[0])
        else:
            geometric_acceptance_scale.append(1)

        if verbose:
            print(f'Scaling the rate by the energy loss and geometric acceptance:\n{scaling_column_name}: {geometric_acceptance_scale}')
    
        volume = df[f'volume_m3_{dm_model}'].iloc[0]
        if verbose:
            print(f'Volume of rock used in acceptance calculations: {volume}')
    
        # Divide by the IceCube volume
        volume_scale.append(volume/(1000**3))
        if verbose:
            print(f'Rates will be scaled by {volume_scaling} to account volume relative to IceCube (1km^3)')

    rate_cms_floating = rate_icecube*geometric_acceptance_scale[0]*volume_scale[0]
    rate_cms_core =     rate_icecube*geometric_acceptance_scale[1]*volume_scale[1]

    return rate_cms_floating, rate_cms_core

convert_from_icecube_to_cms_rates(df, mx=10000, rate_icecube=100000)

In [ ]:
max_rates = []
max_rates_cms_floating = []
max_rates_cms_core     = []

for mass in masses:

    filter = df['mx'] == mass
    dftmp = df[filter]
    dfmax = dftmp.loc[dftmp['rate_1yr'].idxmax()]
    mr = dfmax['rate_1yr']
    max_rates.append(mr)
    mr_float, mr_core = convert_from_icecube_to_cms_rates(dftmp, mx=mass, rate_icecube=mr, ecut=10, hit_id=True)
    max_rates_cms_floating.append(mr_float)
    max_rates_cms_core.append(mr_core)
    
max_rates = np.array(max_rates)
max_rates_cms_floating = np.array(max_rates_cms_floating)
max_rates_cms_core = np.array(max_rates_cms_core)



In [ ]:
plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.plot(masses,max_rates,'o')
plt.xscale('log')
plt.yscale('log')
plt.xlabel(r'm$_{X}$ (GeV/c$^2$)', fontsize=14)
plt.ylabel(r'# at detector / year', fontsize=14)

plt.subplot(1,3,2)
plt.plot(masses,max_rates_cms_floating,'o')
plt.xscale('log')
plt.yscale('log')
plt.xlabel(r'm$_{X}$ (GeV/c$^2$)', fontsize=14)
plt.ylabel(r'# at CMS / year (floating)', fontsize=14)

plt.subplot(1,3,3)
plt.plot(masses,max_rates_cms_core,'o')
plt.xscale('log')
plt.yscale('log')
plt.xlabel(r'm$_{X}$ (GeV/c$^2$)', fontsize=14)
plt.ylabel(r'# at CMS / year (core)', fontsize=14)

plt.tight_layout()
plt.savefig('max_values_rates_icecube_and_cms.png')

# Plot rates as a function of $m_A$ and $\epsilon$.

This replicates Fig. 3 in the original paper. 

In [ ]:
def plot_rates_by_ma_and_epsilon(df, final_state_particles='muons', alphax_assumption='MAX', \
                                live_time='1yr', rate_detector='rate', \
                                rate_conversion=1, rate_label='1 year', \
                                detector_scaling=1, detector_string='IceCube', \
                                detector_min_e_cutoff=100, dm_model='core', \
                                mxs=[10000], convert_to_TeV=False, outdir='./'):

    scaling_column_name = None
    volume = 1
    volume_scaling = 1
    if detector_string=='CMS':
        scaling_column_name = f'frac_ecut{detector_min_e_cutoff:d}_{dm_model}'
        print(f'Scaling the rate by the energy loss and geometric acceptance:\n{scaling_column_name}')

        volume = df[f'volume_m3_{dm_model}'].iloc[0]
        print(f'Volume of rock used in acceptance calculations: {volume}')

        # Divide by the IceCube volume
        volume_scaling = volume/(1000**3)
        print(f'Rates will be scaled by {volume_scaling} to account volume relative to IceCube (1km^3)')

    
    #######################################################
    
    filter =          (df['final_state_particles']==final_state_particles)
    filter = filter & (df['alpha_therm_or_max']==alphax_assumption)
    
    df_tmp = df[filter]
    
    rate_string = f"{rate_detector}_{live_time}"
    print(f"Plotting for rate {rate_string}")
    
    rate = df_tmp[rate_string]
    rate *= rate_conversion # To scale the rate by time

    # Scale by angular acceptance
    scaling = 1
    if scaling_column_name is not None:
        scaling = df_tmp[scaling_column_name] 
    rate *= scaling

    # Scale by volume
    rate *= volume_scaling
    
    # For the plot label
    text = ""
    if detector_string == "CMS":
        text = f'DM model: {dm_model}\n'
    text += f'{detector_string} - {rate_label}'
    
    mx_vals = df_tmp['mx']
    
    x = df_tmp['ma']
    y = df_tmp['epsilon']
    
    #plt.figure(figsize=(16,15))
    plt.figure(figsize=(10,9))
    
    for idx,test_mx in enumerate(mxs):

        scale = [1]
        if scaling_column_name is not None:
            scale = df_tmp[df_tmp['mx']==test_mx][scaling_column_name].values
        
        if len(scale)>0:
            scale = scale[0]
    
        print(f"{idx} {test_mx} {scale}")
    
        plt.subplot(2,2,idx+1)
        
        for expected_rate in [1, 10, 100, 1000]:
    
            # For Icecube
            fractional_test = 0.3
            # For CMS
            #fractional_test = 1.0
            
            filter = (test_mx == mx_vals) & (np.abs(rate-expected_rate)/expected_rate<fractional_test)
            #filter = filter & (df_results['final_state_particles']=='electrons')
            #filter = filter & (df_results['alpha_therm_or_max']=='MAX')
    
            #x = df_tmp['ma']
            #y = df_tmp['epsilon']
            
            plt.plot(x[filter], y[filter], '.', label=f'rate={expected_rate}')
    
            #print(test_mx, expected_rate, df_results[filter]['rate_CMS'])
    
            props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
            plt.gca().text(0.05, 0.15, text, transform=plt.gca().transAxes, 
            fontsize=10, verticalalignment='top', bbox=props)
    
        plt.xlim(0.01, 10)
        plt.ylim(1e-11, 1e-6)
        plt.yscale('log')
        plt.xscale('log')
        plt.xlabel(r"$m_{A'}$ [GeV]", fontsize=18)
        plt.ylabel(r"$\epsilon$", fontsize=18)
    
        plt.legend()

        text_for_mx = f'{int(test_mx):d}'
        units = 'GeV'
        if convert_to_TeV:
            units = 'TeV'
            temp = test_mx/1000
            if temp<1:
                text_for_mx = f'{temp:.1f}'
            else:
                text_for_mx = f'{int(temp):d}'
            
        plt.title(f'$M_X = {text_for_mx}$ {units}', fontsize=18)
    
    #plt.legend()
    plt.tight_layout()
    
    scaling_column_name_string = ""
    if scaling_column_name is not None:
        scaling_column_name_string = f'_{scaling_column_name}'

    print(f'scaling: {scaling_column_name}   {scaling_column_name_string}')
    
    file_string = f"{outdir}/rate_eps_vs_mx_{detector_string}_{rate_label.replace(' ','_')}{scaling_column_name_string}_{dm_model}_{final_state_particles}_alphax_{alphax_assumption}_mX_range_{mxs[0]}_{mxs[-1]}.png"

    print(f'Saving to...{file_string}')
    
    plt.savefig(file_string)

    return 0

### IceCube test

Fig. 3 in [paper](https://arxiv.org/pdf/1509.07525)

In [ ]:
#final_state_particles = 'muons'
final_state_particles = 'electrons'

#alphax_assumption = 'MAX'
alphax_assumption = 'THERMAL'

live_time = '1yr'
rate_detector = 'rate'

####### Time ##########################
#rate_conversion = 1    # 1 year
#rate_label = '1 year'

rate_conversion = 10   # 10 years
rate_label = '10 years'

#rate_conversion = 1/12 # 1 month
#rate_label = '1 month'

####### detector acceptance ##########################

detector_scaling = 1
detector_string = 'IceCube'
scaling_column_name = None
detector_min_e_cutoff = None
dm_model = 'core'

'''
detector_string = 'CMS'
dm_model = 'floating'
#dm_model = 'core'
#dm_model = 'mono-energetic'
detector_min_e_cutoff = 100
'''

mxs = [10, 100, 1000, 1000]

plot_rates_by_ma_and_epsilon(df,
    final_state_particles=final_state_particles, alphax_assumption=alphax_assumption, \
                                live_time=live_time, rate_detector=rate_detector, \
                                rate_conversion=rate_conversion, rate_label=rate_label, \
                                detector_scaling=detector_scaling, detector_string=detector_string, \
                                detector_min_e_cutoff=detector_min_e_cutoff, dm_model=dm_model, \
                                mxs=mxs)



### CMS 

In [ ]:
del df

#infilename = 'rates_muons_electrons_both_alphas_KAPPAS_FOR_TESTING_WITH_CALC_ACCEPTANCES.parquet'
infilename = 'rates_muons_electrons_both_alphas_KAPPAS_10_100_1000_10000_100000_fine_grain_epsilon_and_mas_WITH_CALC_ACCEPTANCES.parquet'

df = pd.read_parquet(infilename)


In [ ]:
df.columns

In [ ]:
print(df['volume_m3_core'].unique())
print(df['volume_m3_floating'].unique())

In [ ]:
final_state_particles = 'muons'
#final_state_particles = 'electrons'

alphax_assumption = 'MAX'
#alphax_assumption = 'THERMAL'

live_time = '1yr'
rate_detector = 'rate'


####### Time ##########################
#rate_conversion = 1    # 1 year
#rate_label = '1 year'

#rate_conversion = 10   # 10 years
#rate_label = '10 years'

rate_conversion = 1/12 # 1 month
rate_label = '1 month'

####### detector acceptance ##########################

#detector_scaling = 1
#detector_string = 'IceCube'
#scaling_column_name = None
#detector_min_e_cutoff = None
#dm_model = 'core'

#'''
detector_string = 'CMS'
scaling_column_name = 'angular_acceptance'
dm_model = 'floating'
#dm_model = 'core'
#dm_model = 'mono-energetic'
detector_min_e_cutoff = 10
#'''

#mxs = [10, 100, 1000, 1000]
#mxs = [100, 1000, 1000, 10000]
mxs = [100, 1000, 10000, 100000]

plot_rates_by_ma_and_epsilon(df,
    final_state_particles=final_state_particles, alphax_assumption=alphax_assumption, \
                                live_time=live_time, rate_detector=rate_detector, \
                                rate_conversion=rate_conversion, rate_label=rate_label, \
                                detector_scaling=detector_scaling, detector_string=detector_string, \
                                detector_min_e_cutoff=detector_min_e_cutoff, dm_model=dm_model, \
                                mxs=mxs, convert_to_TeV=True, outdir='PLOTS_FOR_PHENO_PAPER')



In [ ]:
# Compare with IceCube
final_state_particles = 'muons'
#final_state_particles = 'electrons'

alphax_assumption = 'MAX'
#alphax_assumption = 'THERMAL'

live_time = '1yr'
rate_detector = 'rate'

####### Time ##########################
#rate_conversion = 1    # 1 year
#rate_label = '1 year'

#rate_conversion = 10   # 10 years
#rate_label = '10 years'

rate_conversion = 1/12 # 1 month
rate_label = '1 month'

####### detector acceptance ##########################

detector_scaling = 1
detector_string = 'IceCube'
scaling_column_name = None
detector_min_e_cutoff = None
dm_model = 'core'


#mxs = [10, 100, 1000, 1000]
#mxs = [100, 1000, 1000, 10000]
mxs = [1000, 1000, 10000, 100000]

plot_rates_by_ma_and_epsilon(df,
    final_state_particles=final_state_particles, alphax_assumption=alphax_assumption, \
                                live_time=live_time, rate_detector=rate_detector, \
                                rate_conversion=rate_conversion, rate_label=rate_label, \
                                detector_scaling=detector_scaling, detector_string=detector_string, \
                                detector_min_e_cutoff=detector_min_e_cutoff, dm_model=dm_model, \
                                mxs=mxs)



# Other plots

# Parameters and terms

## Parameters

* $m_X$ mass of DM particles
* $m_{A'}$ mass of dark photon
* $\epsilon$ (CHECK THIS, KINEMATIC MIXING?)
* $\alpha_X$ dark matter strong coupling term

## Dependent function

The number density for the dark matter particles $N_X$ appears in `singleElementCap` and is $0.3/M_X

`CCap` (usually `cap1` or `cap2`) $C_{\rm cap}$ is a function of ($m_X, m_{A'}, \epsilon, \alpha, \alpha_X$) "Total Capture Rate: The total capture rate, Eq. (3.5), sums the single-element capture
rates over the ten most abundant elements in Earth"

Sometimes CCap is written as a function of $\kappa_0$ (`kappa_0`) which iis a function $M_X and \alpha$,

$\sigma_{\rm ann}$ `sigma`, `sigmaVtree` ($m_X, m_{A'}, \alpha_X$)- "Tree-level Annihilation Cross Section: The annihilation cross section Eq. (3.19) without
Sommerfeld enhancement is..."

`sommerfeld` ($v, m_X, m_{A'}, \alpha_X$) $S_S$ "Sommerfeld Enhancement: The non-perturbative Sommerfeld enhancement Eq. (3.20)
factor that boosts the tree-level annihilation rate in the presence of a long-range force is a
function of velocity, dark matter mass, mediator mass, and the dark fine structure constant,"

Our factor `sommerfeld`  $\langle S_S\rangle$is "Thermally Averaged Sommerfeld Enhancement: The thermally averaged s-wave Sommerfeld enhancement Eq. (3.21) is" and comes from `thermAvgSommerfeld` and is a function of ($m_X, m_{A'}, \alpha_X$)

`ann`or $C_{\rm ann}$ ($m_X, \sigma_{\rm ann}v, \langle S_S\rangle$) comes from `cAnn` "Annihilation Rate: Defines the annihilation rate for thermalized dark matter at the center
of the Earth, Eq. (3.17)."

`gammaAnn` is a function of $C_{\rm cap}$ and $C_{\rm ann}$ and is "Annihilation Rate: Defines the rate at which the captured dark matter population decreases due to annihilations, Eq. (3.3)"

`tau` depends on $C_{\rm cap}$ and $C_{\rm ann}$ and is "Equilibrium Time: The equilibrium time τ , Eq. (3.22), is the characteristic time for the
dark matter capture and annihilation processes to equilibrate."

$L$ is the `decayLength` and is a function of ($m_X, m_{A}, \epsilon, B$) where $B$ is the branching ratio. 

`Edecay` comes from `epsilonDecay` and is $\epsilon_{\rm decay}$ and is Decay Parameter: The decay parameter, Eq. (3.27), represents the probability that a
dark photon produced at the center of Earth propagates to a distance R⊕ within a detector’s
of effective depth D of the surface. For IceCube, D ≈ 1 km [1]. The quantity L is the
characteristic decay length of the dark photon.

In [ ]:
############################################
def find_index_of_closest_value(values_to_look_for, arr):

    closest_values = []
    indices = []
    
    for v in values_to_look_for:
        diff = np.abs(arr - v)
        
        min_diff = min(diff)
        
        idx = diff.to_list().index(min_diff)
        
        print(idx, min_diff, arr.iloc[idx])

        closest_values.append(arr.iloc[idx])
        indices.append(idx)

    return indices, closest_values

###########################################################################


def make_summary_plot(df, final_state_particles='muons', \
                      alphax_assumption = 'MAX', \
                      xvar = 'mx', \
                      yvar = 'sigma', \
                      xlabel=None, \
                      ylabel=None, \
                      extra_tag='DEFAULT', \
                      epsilons=None, \
                      ma_vals_to_plot=None, \
                      legend_loc=None, \
                     ):
        
    filter = (df['final_state_particles']==final_state_particles)
    filter = filter & (df['alpha_therm_or_max']==alphax_assumption)
    
    df_tmp = df[filter]
    
    x = df_tmp[xvar]
    
    y = df_tmp[yvar]


    if xlabel is None:
        xlabel = xvar
    if ylabel is None:
        ylabel = yvar

    ma_vals = df_tmp['ma']

    mas = ma_vals.unique()
    print(len(mas), mas)

    if ma_vals_to_plot is None:
        ma_vals_to_plot = [0.230, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]

    ma_tag = 'ma_threshold_to_0.7'
    
    indices, closest_vals = find_index_of_closest_value(ma_vals_to_plot, ma_vals)
    mas = closest_vals
    print("found closest values")
    print(closest_vals)
    
    # For finer-grained estimates
    mxs = df['mx'].unique()
    
    if epsilons is None:
        epsilons = [1e-11, 1e-10, 1e-9, 1e-8, 1e-7, 1e-6 ]
    
    print("\nepsilons: ")
    print(epsilons)
    print()

    ##############################################################
    #
    ##############################################################
    if len(epsilons)==6:
        plt.figure(figsize=(16,6))
    elif len(epsilons)==4:
        plt.figure(figsize=(8,8))
    elif len(epsilons)==1:
        plt.figure(figsize=(6,6))
    else:
        plt.figure(figsize=(16,15))
    
    
    for idx,eps in enumerate(epsilons):
    
        #print(f"{idx} {eps}")
    
        #plt.subplot(6,3,idx+1)
        if len(epsilons)==6:
            plt.subplot(2,3,idx+1)
        elif len(epsilons)==4:
            plt.subplot(2,2,idx+1)
        elif len(epsilons)==1:
            plt.subplot(1,1,idx+1)
        else:
            plt.subplot(6,3,idx+1)
        
        for ma in mas:
            
            filter = (df_tmp['ma']== ma) & (df_tmp['epsilon']==eps)
            
            plt.plot(x[filter], y[filter], '.-', label=f'ma={ma:.3f}')
    
            #print(test_mx, expected_rate, df_results[filter]['rate_CMS'])
    
        #plt.xlim()
        if yvar=='n_particles':
            plt.ylim(1e25,1e36)
        elif yvar=='sigma':
            plt.ylim(1e-10, 1e-2)
        elif yvar=='gammaAnn':
            plt.ylim(1, 1e15)
        elif yvar=='L':
            plt.ylim(1, 1e15)
        elif yvar=='ann':
            plt.ylim(1e-50, 1e-25)
        elif yvar=='rate_CMS_1month':
            plt.ylim(1, 1e4)
        elif yvar=='tau':
            plt.ylim(1e8, 1e25)
        elif yvar=='Edecay':
            plt.ylim(1e-10, 1)
    
        
        if len(y[filter])>0:
            plt.yscale('log')
            plt.xscale('log')
        
        plt.xlabel(f'{xvar}', fontsize=18)
        plt.ylabel(f"{yvar}", fontsize=18)

        if legend_loc is not None:
            plt.legend(loc=legend_loc)
        else:
            plt.legend()

        plt.xlabel(xlabel)
        plt.ylabel(ylabel)
    
        plt.title(f'$\epsilon = {eps:.2e}$', fontsize=18)
    
    #plt.legend()
    plt.tight_layout()
    
    file_string = f"{yvar}_vs_{xvar}_{final_state_particles}_alphax_{alphax_assumption}_{extra_tag}.png"
    plt.savefig(file_string)

    del df_tmp

#################################################################
#make_summary_plot()

In [ ]:
if 'df' in locals():
    print("Dataframe exists! Deleting to free up memory before reading in a new one")
    del df
    

#infilename = 'rates_muons_electrons_both_alphas_KAPPAS_FOR_TESTING_WITH_CALC_ACCEPTANCES.parquet'
#infilename = 'rates_muons_electrons_both_alphas_KAPPAS_10_100_1000_10000_100000_fine_grain_epsilon_and_mas_WITH_CALC_ACCEPTANCES.parquet'
#infilename = 'rates_muons_electrons_both_alphas_KAPPAS_10_1000000_varying_steps_coarse_grain_epsilon_and_mas_WITH_CALC_ACCEPTANCES.parquet'
infilename = 'rates_muons_electrons_both_alphas_KAPPAS_10_1000000_varying_steps_coarse_grain_epsilon_and_mas_WITH_CALC_ACCEPTANCES.parquet'

df = pd.read_parquet(infilename)

print(df.columns)
print()
print(df['mx'].unique())

In [ ]:
epsilons = [1e-10, 1e-9, 1e-8, 1e-7]
ma_vals_to_plot = [0.23, 0.25, 0.3, 0.4]

make_summary_plot(df, yvar='gammaAnn', epsilons=epsilons, ma_vals_to_plot=ma_vals_to_plot, extra_tag='TESTING')


In [ ]:
epsilons = [1e-10, 1e-9, 1e-8, 1e-7]
ma_vals_to_plot = [0.23, 0.25, 0.3, 0.4]

make_summary_plot(df, yvar='Edecay', \
                  ylabel='Probability of dark photon propagating to...', xlabel=r'M$_{DM}$ (GeV/c$^2$)', \
                  epsilons=epsilons, ma_vals_to_plot=ma_vals_to_plot, extra_tag='TESTING')


In [ ]:
ma_vals_to_plot = [0.23, 0.25, 0.3, 0.4]
epsilons = [1e-10, 1e-9, 1e-8, 1e-7]

make_summary_plot(df,yvar='n_particles', epsilons=epsilons, ma_vals_to_plot=ma_vals_to_plot, \
                  ylabel='# accum DM particles', xlabel=r'M$_{DM}$ (GeV/c$^2$)', legend_loc='lower right', \
                extra_tag='')


In [ ]:
ma_vals_to_plot = [0.23, 0.25, 0.3, 0.4]
epsilons = [1e-10, 1e-9, 1e-8, 1e-7]


make_summary_plot(df,yvar='L', epsilons=epsilons, ma_vals_to_plot=ma_vals_to_plot, \
                  ylabel='Decay length (cm)', xlabel=r'M$_{DM}$ (GeV/c$^2$)', \
                  extra_tag='', legend_loc='lower right')


In [ ]:
ma_vals_to_plot = [0.23, 0.25, 0.3, 0.4]
epsilons = [1e-10]


make_summary_plot(df,yvar='sigma', epsilons=epsilons, ma_vals_to_plot=ma_vals_to_plot, \
                  ylabel='sigma', xlabel=r'M$_{DM}$ (GeV/c$^2$)', \
                  extra_tag='', legend_loc='lower right')


In [ ]:
ma_vals_to_plot = [0.23, 0.25, 0.3, 0.4]
epsilons = [1e-10]


make_summary_plot(df,yvar='sommerfeld', epsilons=epsilons, ma_vals_to_plot=ma_vals_to_plot, \
                  ylabel='Sommerfeld', xlabel=r'M$_{DM}$ (GeV/c$^2$)', \
                  extra_tag='', legend_loc='lower right')
